In [17]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.func import stack_module_state, functional_call, hessian
from scipy.linalg import solve
import numpy as np
import random


SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def true_function(x):
    return torch.sin(2.0 * x) + 0.3 * torch.cos(5.0 * x)


def make_dataset(n_train=256, n_test=256, noise_std=0.1):
    x_train = torch.linspace(-3.0, 3.0, n_train).unsqueeze(1)
    y_clean_train = true_function(x_train)
    y_train = y_clean_train + noise_std * torch.randn_like(y_clean_train)

    x_test = torch.linspace(-3.2, 3.2, n_test).unsqueeze(1)
    y_clean_test = true_function(x_test)
    y_test = y_clean_test + noise_std * torch.randn_like(y_clean_test)

    return (
        x_train.to(device),
        y_train.to(device),
        x_test.to(device),
        y_test.to(device),
        y_clean_test.to(device),
    )


class BRANN(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, output_dim)
        )
        self.alpha = 1.0  # априорная точность
        self.beta = 1.0   # точность шума

    def forward(self, x):
        return self.net(x)


    def compute_effective_params(self, X, y):
        """
        Точное вычисление эффективного числа параметров γ через полный Гессиан.
        Работает с любой архитектурой (Sequential, вложенные модули и т.д.)
        """
        device = next(self.parameters()).device
        X, y = X.to(device), y.to(device)
        
        # 1. Собираем все веса в один плоский вектор
        # Это нужно, чтобы вычислить Гессиан размерности (N_weights x N_weights)
        def get_flat_params(model):
            return torch.cat([p.flatten() for p in model.parameters()])
        
        def set_flat_params(model, flat_params):
            offset = 0
            for p in model.parameters():
                numel = p.numel()
                p.data.copy_(flat_params[offset:offset+numel].view_as(p))
                offset += numel
        
        # Текущие веса (чтобы вернуться к ним после вычислений)
        original_params = get_flat_params(self).clone().detach()
        n_weights = len(original_params)
        
        # 2. Функция потерь для автоматического дифференцирования
        # Она принимает плоский вектор весов, загружает их в модель и считает ошибку
        def loss_fn(flat_w):
            set_flat_params(self, flat_w)
            output = self(X)
            # 0.5 * MSE, как в классической формуле Байесовской регуляризации
            return 0.5 * torch.mean((output - y) ** 2)
        
        # 3. Вычисляем точный Гессиан: H = d²L/dw²
        # torch.autograd.functional.hessian сам разберется с графом вычислений
        try:
            H = torch.autograd.functional.hessian(loss_fn, original_params)
        except RuntimeError as e:
            print(f"Warning: Hessian computation failed ({e}), using diagonal approximation")
            # Fallback на диагональное приближение, если не хватило памяти
            return self.compute_effective_params_diag(X, y)
        
        # 4. Регуляризованный Гессиан: G = β·H + α·I
        G = self.beta * H + self.alpha * torch.eye(n_weights, device=device)
        
        # 5. Вычисляем trace(G⁻¹)
        try:
            # Пытаемся обратить матрицу точно
            G_inv = torch.linalg.inv(G)
            trace_inv = torch.trace(G_inv).item()
        except RuntimeError:
            # Если матрица вырождена, используем псевдообращение
            G_inv = torch.linalg.pinv(G)
            trace_inv = torch.trace(G_inv).item()
        
        # 6. Формула Маккея (уравнение 28 из статьи)
        gamma = n_weights - self.alpha * trace_inv
        
        # Возвращаем параметры модели в исходное состояние (на всякий случай)
        set_flat_params(self, original_params)
        
        return max(1e-6, min(gamma, n_weights))


    def update_hyperparameters(self, X, y, W_norm):
        gamma = self.compute_effective_params(X, y)
        N = y.size(0)
        ed = 0.5 * ((self(X) - y)**2).mean().item() * N
        ew = 0.5 * W_norm.item()
        
        device = next(self.parameters()).device
        X, y = X.to(device), y.to(device)
    
        
        # 2. Вычисляем γ (если не передан)
        if gamma is None:
            gamma = self.compute_effective_params_diag(X, y)  # используем стабильную диагональную аппроксимацию
        
        n_data = len(X)
        n_weights = sum(p.numel() for p in self.parameters())
        
        # 3. Защита γ от выхода за разумные пределы
        gamma = max(1e-6, min(gamma, n_weights - 1e-6))
        
        # 4. Новые значения α и β
        alpha_new = gamma / (2 * ew + 1e-10)
        beta_new = (n_data - gamma) / (2 * ed + 1e-10)
        
        # 5. Плавное обновление (экспоненциальное сглаживание)
        # Это предотвращает резкие скачки
        momentum = 0.9  # чем больше, тем плавнее изменения
        self.alpha = momentum * self.alpha + (1 - momentum) * alpha_new
        self.beta = momentum * self.beta + (1 - momentum) * beta_new
    
        # 6. Жесткие ограничения (физически обоснованные)
        self.alpha = np.clip(self.alpha, 1e-6, 100.0)   # α не должен быть 0 или бесконечностью
        self.beta = np.clip(self.beta, 1e-2, 1e4)       # β в разумных пределах
        
        return self.alpha, self.beta

# --- Цикл обучения ---


In [18]:
model = BRANN(10, 20, 1)
optimizer = optim.LBFGS(model.parameters(), lr=0.001, max_iter=20)

X = torch.randn(100, 10)
y = torch.randn(100, 1)

for epoch in range(150):
    def closure():
        optimizer.zero_grad()
        loss = model.beta * 0.5 * ((model(X) - y)**2).mean() + \
               model.alpha * 0.5 * sum(p.pow(2).sum() for p in model.parameters())
        loss.backward()
        return loss

    optimizer.step(closure)
    
    # Обновление гиперпараметров каждые несколько шагов
    if epoch % 3 == 0:
        W_norm = sum(p.pow(2).sum() for p in model.parameters())
        a, b = model.update_hyperparameters(X, y, W_norm)
        print(f"Epoch {epoch}: alpha={a:.3f}, beta={b:.3f}")

Epoch 0: alpha=0.900, beta=0.982
Epoch 3: alpha=0.810, beta=0.966
Epoch 6: alpha=0.729, beta=0.952
Epoch 9: alpha=0.656, beta=0.941
Epoch 12: alpha=0.590, beta=0.931
Epoch 15: alpha=0.531, beta=0.922
Epoch 18: alpha=0.478, beta=0.915
Epoch 21: alpha=0.430, beta=0.908
Epoch 24: alpha=0.387, beta=0.903
Epoch 27: alpha=0.349, beta=0.899
Epoch 30: alpha=0.314, beta=0.895
Epoch 33: alpha=0.282, beta=0.892
Epoch 36: alpha=0.254, beta=0.889
Epoch 39: alpha=0.229, beta=0.887
Epoch 42: alpha=0.206, beta=0.886
Epoch 45: alpha=0.185, beta=0.886
Epoch 48: alpha=0.167, beta=0.886
Epoch 51: alpha=0.150, beta=0.888
Epoch 54: alpha=0.135, beta=0.890
Epoch 57: alpha=0.122, beta=0.892
Epoch 60: alpha=0.109, beta=0.895
Epoch 63: alpha=0.098, beta=0.898
Epoch 66: alpha=0.089, beta=0.901
Epoch 69: alpha=0.080, beta=0.904
Epoch 72: alpha=0.072, beta=0.907
Epoch 75: alpha=0.065, beta=0.911
Epoch 78: alpha=0.058, beta=0.915
Epoch 81: alpha=0.052, beta=0.918
Epoch 84: alpha=0.047, beta=0.924
Epoch 87: alpha=0.